# 00 — Data Preparation & Feature Engineering

Pipeline de ETL que integra 5 fontes de dados corporativos (dados gerais de RH, pesquisa de clima, avaliação gerencial, registros de ponto) em um dataset analítico unificado.

**Fontes de dados:**
- `general_data.csv` — Dados demográficos, cargo, salário, histórico profissional
- `employee_survey_data.csv` — Pesquisa de clima organizacional
- `manager_survey_data.csv` — Avaliação gerencial de performance e envolvimento
- `in_time.csv` / `out_time.csv` — Registros de ponto (entrada/saída) de 2015

**Output:** `master_dataset.csv` com todas as features integradas, prontas para análise.

---
## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Visual config
plt.style.use('dark_background')
COLORS = {
    'primary': '#6366F1',
    'secondary': '#8B5CF6',
    'accent': '#EC4899',
    'success': '#10B981',
    'warning': '#F59E0B',
    'danger': '#EF4444',
    'text': '#E2E8F0',
    'muted': '#94A3B8',
    'bg': '#0F172A',
    'card': '#1E293B'
}

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [2]:
# Load all datasets
DATA_PATH = '../data/raw/'

df_general = pd.read_csv(f'{DATA_PATH}general_data.csv')
df_survey = pd.read_csv(f'{DATA_PATH}employee_survey_data.csv')
df_manager = pd.read_csv(f'{DATA_PATH}manager_survey_data.csv')
df_intime = pd.read_csv(f'{DATA_PATH}in_time.csv')
df_outtime = pd.read_csv(f'{DATA_PATH}out_time.csv')

print(f'General Data:     {df_general.shape[0]:,} rows × {df_general.shape[1]} cols')
print(f'Employee Survey:  {df_survey.shape[0]:,} rows × {df_survey.shape[1]} cols')
print(f'Manager Survey:   {df_manager.shape[0]:,} rows × {df_manager.shape[1]} cols')
print(f'In Time:          {df_intime.shape[0]:,} rows × {df_intime.shape[1]} cols')
print(f'Out Time:         {df_outtime.shape[0]:,} rows × {df_outtime.shape[1]} cols')

General Data:     4,410 rows × 24 cols
Employee Survey:  4,410 rows × 4 cols
Manager Survey:   4,410 rows × 3 cols
In Time:          4,410 rows × 262 cols
Out Time:         4,410 rows × 262 cols


---
## 2. Data Validation & Integrity Checks

In [3]:
# Validate EmployeeID consistency across all datasets
general_ids = set(df_general['EmployeeID'])
survey_ids = set(df_survey['EmployeeID'])
manager_ids = set(df_manager['EmployeeID'])

# in_time/out_time use row index as EmployeeID ("Unnamed: 0" column)
df_intime = df_intime.rename(columns={'Unnamed: 0': 'EmployeeID'})
df_outtime = df_outtime.rename(columns={'Unnamed: 0': 'EmployeeID'})
intime_ids = set(df_intime['EmployeeID'])
outtime_ids = set(df_outtime['EmployeeID'])

print('=== Employee ID Consistency ===')
print(f'General Data IDs:     {len(general_ids):,}')
print(f'Employee Survey IDs:  {len(survey_ids):,}')
print(f'Manager Survey IDs:   {len(manager_ids):,}')
print(f'In Time IDs:          {len(intime_ids):,}')
print(f'Out Time IDs:         {len(outtime_ids):,}')
print()

# Check for perfect overlap
all_match = (general_ids == survey_ids == manager_ids == intime_ids == outtime_ids)
print(f'All datasets share the same EmployeeIDs: {all_match}')

if not all_match:
    common = general_ids & survey_ids & manager_ids & intime_ids & outtime_ids
    print(f'Common IDs across all datasets: {len(common):,}')

=== Employee ID Consistency ===
General Data IDs:     4,410
Employee Survey IDs:  4,410
Manager Survey IDs:   4,410
In Time IDs:          4,410
Out Time IDs:         4,410

All datasets share the same EmployeeIDs: True


In [4]:
# Check general_data overview
print('=== General Data Overview ===')
print(f'\nShape: {df_general.shape}')
print(f'\nDuplicate rows: {df_general.duplicated().sum()}')
print(f'Duplicate EmployeeIDs: {df_general["EmployeeID"].duplicated().sum()}')
print(f'\n--- Missing Values ---')
missing = df_general.isnull().sum()
missing_pct = (missing / len(df_general) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Pct (%)': missing_pct})
print(missing_df[missing_df['Count'] > 0].to_string())
print(f'\n--- Data Types ---')
print(df_general.dtypes.to_string())

=== General Data Overview ===

Shape: (4410, 24)

Duplicate rows: 0
Duplicate EmployeeIDs: 0

--- Missing Values ---
                    Count  Pct (%)
NumCompaniesWorked     19     0.43
TotalWorkingYears       9     0.20

--- Data Types ---
Age                          int64
Attrition                   object
BusinessTravel              object
Department                  object
DistanceFromHome             int64
Education                    int64
EducationField              object
EmployeeCount                int64
EmployeeID                   int64
Gender                      object
JobLevel                     int64
JobRole                     object
MaritalStatus               object
MonthlyIncome                int64
NumCompaniesWorked         float64
Over18                      object
PercentSalaryHike            int64
StandardHours                int64
StockOptionLevel             int64
TotalWorkingYears          float64
TrainingTimesLastYear        int64
YearsAtCompany         

In [5]:
# Check constant/uninformative columns
print('=== Constant / Low-Variance Columns ===')
for col in df_general.columns:
    nunique = df_general[col].nunique()
    if nunique <= 2:
        print(f'{col}: {nunique} unique values → {df_general[col].unique()}')

=== Constant / Low-Variance Columns ===
Attrition: 2 unique values → ['No' 'Yes']
EmployeeCount: 1 unique values → [1]
Gender: 2 unique values → ['Female' 'Male']
Over18: 1 unique values → ['Y']
StandardHours: 1 unique values → [8]


In [6]:
# Survey data overview
print('=== Employee Survey Data ===')
print(df_survey.describe().to_string())
print(f'\nMissing values: {df_survey.isnull().sum().sum()}')

print('\n=== Manager Survey Data ===')
print(df_manager.describe().to_string())
print(f'\nMissing values: {df_manager.isnull().sum().sum()}')

=== Employee Survey Data ===
        EmployeeID  EnvironmentSatisfaction  JobSatisfaction  WorkLifeBalance
count  4410.000000              4385.000000      4390.000000      4372.000000
mean   2205.500000                 2.723603         2.728246         2.761436
std    1273.201673                 1.092756         1.101253         0.706245
min       1.000000                 1.000000         1.000000         1.000000
25%    1103.250000                 2.000000         2.000000         2.000000
50%    2205.500000                 3.000000         3.000000         3.000000
75%    3307.750000                 4.000000         4.000000         3.000000
max    4410.000000                 4.000000         4.000000         4.000000

Missing values: 83

=== Manager Survey Data ===
        EmployeeID  JobInvolvement  PerformanceRating
count  4410.000000     4410.000000        4410.000000
mean   2205.500000        2.729932           3.153741
std    1273.201673        0.711400           0.360742
min 

---
## 3. Merge Core HR Datasets

In [7]:
# Merge general_data + employee_survey + manager_survey
df = df_general.merge(df_survey, on='EmployeeID', how='left')
df = df.merge(df_manager, on='EmployeeID', how='left')

print(f'Merged dataset: {df.shape[0]:,} rows × {df.shape[1]} cols')
print(f'\nNew columns from survey: EnvironmentSatisfaction, JobSatisfaction, WorkLifeBalance')
print(f'New columns from manager: JobInvolvement, PerformanceRating')

# Drop constant/uninformative columns
cols_to_drop = ['EmployeeCount', 'Over18', 'StandardHours']
df = df.drop(columns=cols_to_drop)
print(f'\nDropped constant columns: {cols_to_drop}')
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} cols')

Merged dataset: 4,410 rows × 29 cols

New columns from survey: EnvironmentSatisfaction, JobSatisfaction, WorkLifeBalance
New columns from manager: JobInvolvement, PerformanceRating

Dropped constant columns: ['EmployeeCount', 'Over18', 'StandardHours']
Final shape: 4,410 rows × 26 cols


In [8]:
# Quick validation of the merge
print('=== Merged Dataset Sample ===')
df.head(3)

=== Merged Dataset Sample ===


,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeID,Gender,JobLevel,...,TotalWorkingYears,TrainingTimesLastYear,YearsAtCompany,YearsSinceLastPromotion,YearsWithCurrManager,EnvironmentSatisfaction,JobSatisfaction,WorkLifeBalance,JobInvolvement,PerformanceRating
0,51,No,Travel_Rarely,Sales,6,2,Life Sciences,1,Female,1,...,1.0,6,1,0,0,3.0,4.0,2.0,3,3
1,31,Yes,Travel_Frequently,Research & Development,10,1,Life Sciences,2,Female,1,...,6.0,3,5,1,4,3.0,2.0,4.0,2,4
2,32,No,Travel_Frequently,Research & Development,17,4,Other,3,Male,4,...,5.0,2,5,0,3,2.0,2.0,1.0,3,3


---
## 4. Time & Attendance Data Processing

The `in_time` and `out_time` datasets contain daily check-in/check-out timestamps for all 4,410 employees across 261 working days in 2015. We'll transform this wide-format data into meaningful per-employee attendance metrics.

In [9]:
# Identify date columns (all columns except EmployeeID)
date_cols = [col for col in df_intime.columns if col != 'EmployeeID']
print(f'Working days in dataset: {len(date_cols)}')
print(f'Date range: {date_cols[0]} to {date_cols[-1]}')

Working days in dataset: 261
Date range: 2015-01-01 to 2015-12-31


In [10]:
# Melt from wide to long format
df_in_long = df_intime.melt(
    id_vars=['EmployeeID'],
    value_vars=date_cols,
    var_name='Date',
    value_name='InTime'
)

df_out_long = df_outtime.melt(
    id_vars=['EmployeeID'],
    value_vars=date_cols,
    var_name='Date',
    value_name='OutTime'
)

# Merge in/out events
df_events = df_in_long.merge(df_out_long, on=['EmployeeID', 'Date'], how='outer')

# Convert to datetime
df_events['Date'] = pd.to_datetime(df_events['Date'])
df_events['InTime'] = pd.to_datetime(df_events['InTime'])
df_events['OutTime'] = pd.to_datetime(df_events['OutTime'])

print(f'Events dataset: {df_events.shape[0]:,} rows (employees × days)')
df_events.head()

Events dataset: 1,151,010 rows (employees × days)


,EmployeeID,Date,InTime,OutTime
0,1,2015-01-01,NaT,NaT
1,2,2015-01-01,NaT,NaT
2,3,2015-01-01,NaT,NaT
3,4,2015-01-01,NaT,NaT
4,5,2015-01-01,NaT,NaT


In [11]:
# Identify non-working days (holidays) — days where ALL employees have NaT
daily_presence = df_events.groupby('Date')['InTime'].apply(lambda x: x.notna().sum())
non_working_days = daily_presence[daily_presence == 0].index.tolist()

print(f'Non-working days (holidays/company days off): {len(non_working_days)}')
for d in non_working_days:
    print(f'  - {d.strftime("%Y-%m-%d")} ({d.strftime("%A")})')

Non-working days (holidays/company days off): 12
  - 2015-01-01 (Thursday)
  - 2015-01-14 (Wednesday)
  - 2015-01-26 (Monday)
  - 2015-03-05 (Thursday)
  - 2015-05-01 (Friday)
  - 2015-07-17 (Friday)
  - 2015-09-17 (Thursday)
  - 2015-10-02 (Friday)
  - 2015-11-09 (Monday)
  - 2015-11-10 (Tuesday)
  - 2015-11-11 (Wednesday)
  - 2015-12-25 (Friday)


In [12]:
# Remove non-working days from the events dataset
df_events = df_events[~df_events['Date'].isin(non_working_days)].copy()
total_working_days = df_events['Date'].nunique()
print(f'Working days after removing holidays: {total_working_days}')
print(f'Events after filtering: {df_events.shape[0]:,}')

Working days after removing holidays: 249
Events after filtering: 1,098,090


In [13]:
# Calculate work hours and attendance flags
df_events['WorkHours'] = (df_events['OutTime'] - df_events['InTime']).dt.total_seconds() / 3600

# Attendance flags
df_events['Present'] = df_events['InTime'].notna().astype(int)
df_events['Absent'] = df_events['InTime'].isna().astype(int)

# Arrival/Departure hour (decimal)
df_events['ArrivalHour'] = df_events['InTime'].dt.hour + df_events['InTime'].dt.minute / 60
df_events['DepartureHour'] = df_events['OutTime'].dt.hour + df_events['OutTime'].dt.minute / 60

# Work pattern flags (based on standard 8h workday)
df_events['ShortDay'] = (df_events['WorkHours'] < 7).astype(int)  # < 7h = short day
df_events['LongDay'] = (df_events['WorkHours'] > 9).astype(int)   # > 9h = long day
df_events['OvertimeHours'] = (df_events['WorkHours'] - 8).clip(lower=0)  # hours beyond standard

# Month for seasonal analysis
df_events['Month'] = df_events['Date'].dt.month
df_events['DayOfWeek'] = df_events['Date'].dt.day_name()

print('=== Work Hours Statistics ===')
print(df_events['WorkHours'].describe().round(2).to_string())
print(f'\nPresent days: {df_events["Present"].sum():,}')
print(f'Absent days:  {df_events["Absent"].sum():,}')
print(f'Short days:   {df_events["ShortDay"].sum():,}')
print(f'Long days:    {df_events["LongDay"].sum():,}')

=== Work Hours Statistics ===
count    1041930.00
mean           7.71
std            1.38
min            4.73
25%            6.66
50%            7.42
75%            8.40
max           12.09

Present days: 1,041,930
Absent days:  56,160
Short days:   376,220
Long days:    195,372


In [14]:
# Check for anomalies: in-only or out-only records
in_only = df_events['InTime'].notna() & df_events['OutTime'].isna()
out_only = df_events['OutTime'].notna() & df_events['InTime'].isna()

print(f'Records with InTime only (no OutTime): {in_only.sum()}')
print(f'Records with OutTime only (no InTime): {out_only.sum()}')

# Check for negative or extreme work hours
negative = (df_events['WorkHours'] < 0).sum()
extreme = (df_events['WorkHours'] > 16).sum()
print(f'Negative work hours: {negative}')
print(f'Extreme work hours (>16h): {extreme}')

Records with InTime only (no OutTime): 0
Records with OutTime only (no InTime): 0
Negative work hours: 0
Extreme work hours (>16h): 0


---
## 5. Feature Engineering — Employee-Level Attendance Metrics

Aggregating daily events into per-employee attendance and behavior features.

In [15]:
# Aggregate attendance metrics per employee
df_attendance = df_events.groupby('EmployeeID').agg(
    # Work hours
    AvgDailyWorkHours=('WorkHours', 'mean'),
    StdDailyWorkHours=('WorkHours', 'std'),
    MedianDailyWorkHours=('WorkHours', 'median'),
    
    # Attendance
    TotalPresent=('Present', 'sum'),
    TotalAbsent=('Absent', 'sum'),
    
    # Work pattern
    TotalShortDays=('ShortDay', 'sum'),
    TotalLongDays=('LongDay', 'sum'),
    TotalOvertimeHours=('OvertimeHours', 'sum'),
    
    # Punctuality
    AvgArrivalHour=('ArrivalHour', 'mean'),
    StdArrivalHour=('ArrivalHour', 'std'),
    AvgDepartureHour=('DepartureHour', 'mean'),
    StdDepartureHour=('DepartureHour', 'std'),
).reset_index()

# Calculated metrics
df_attendance['AbsenceRate'] = (
    df_attendance['TotalAbsent'] / total_working_days * 100
).round(2)

df_attendance['AvgOvertimeHoursPerDay'] = (
    df_attendance['TotalOvertimeHours'] / df_attendance['TotalPresent']
).round(2)

df_attendance['ShortDayRate'] = (
    df_attendance['TotalShortDays'] / df_attendance['TotalPresent'] * 100
).round(2)

df_attendance['LongDayRate'] = (
    df_attendance['TotalLongDays'] / df_attendance['TotalPresent'] * 100
).round(2)

print(f'Attendance features: {df_attendance.shape[1] - 1} per employee')
print(f'\n=== Attendance Metrics Summary ===')
print(df_attendance.describe().round(2).to_string())

Attendance features: 16 per employee

=== Attendance Metrics Summary ===
       EmployeeID  AvgDailyWorkHours  StdDailyWorkHours  MedianDailyWorkHours  TotalPresent  TotalAbsent  TotalShortDays  TotalLongDays  TotalOvertimeHours  AvgArrivalHour  StdArrivalHour  AvgDepartureHour  StdDepartureHour  AbsenceRate  AvgOvertimeHoursPerDay  ShortDayRate  LongDayRate
count     4410.00            4410.00            4410.00               4410.00       4410.00      4410.00         4410.00        4410.00             4410.00         4410.00         4410.00           4410.00           4410.00      4410.00                 4410.00       4410.00      4410.00
mean      2205.50               7.70               0.30                  7.70        236.27        12.73           85.31          44.30              104.93            9.99            0.28             17.69              0.41         5.11                    0.44         36.31        18.57
std       1273.20               1.34               0.01        

In [16]:
# Monthly engagement trend — detect declining/increasing engagement over the year
monthly_hours = df_events.groupby(['EmployeeID', 'Month'])['WorkHours'].mean().reset_index()
monthly_hours.columns = ['EmployeeID', 'Month', 'MonthlyAvgHours']

# Calculate slope of monthly hours (trend) for each employee
from scipy.stats import linregress

def calc_trend(group):
    if len(group) < 3:
        return pd.Series({'EngagementTrend': 0, 'TrendPValue': 1})
    slope, _, _, p_value, _ = linregress(group['Month'], group['MonthlyAvgHours'])
    return pd.Series({'EngagementTrend': round(slope, 4), 'TrendPValue': round(p_value, 4)})

df_trend = monthly_hours.groupby('EmployeeID').apply(calc_trend).reset_index()

# Classify trend
conditions = [
    (df_trend['EngagementTrend'] > 0.05) & (df_trend['TrendPValue'] < 0.1),
    (df_trend['EngagementTrend'] < -0.05) & (df_trend['TrendPValue'] < 0.1),
]
choices = ['Increasing', 'Declining']
df_trend['TrendCategory'] = np.select(conditions, choices, default='Stable')

print('=== Engagement Trend Distribution ===')
print(df_trend['TrendCategory'].value_counts().to_string())

# Merge with attendance
df_attendance = df_attendance.merge(df_trend[['EmployeeID', 'EngagementTrend', 'TrendCategory']], on='EmployeeID', how='left')

=== Engagement Trend Distribution ===
TrendCategory
Stable    4410


---
## 6. Missing Values Treatment

In [17]:
# Check missing values in the merged core dataset
print('=== Missing Values in Core Dataset ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing': missing, 'Pct (%)': missing_pct})
print(missing_report[missing_report['Missing'] > 0].to_string())

=== Missing Values in Core Dataset ===
                         Missing  Pct (%)
NumCompaniesWorked            19     0.43
TotalWorkingYears              9     0.20
EnvironmentSatisfaction       25     0.57
JobSatisfaction               20     0.45
WorkLifeBalance               38     0.86


In [18]:
# Fill missing values with median (numeric columns with low missingness)
# NumCompaniesWorked: 19 missing (0.43%) → median imputation
# TotalWorkingYears: 9 missing (0.20%) → median imputation

for col in ['NumCompaniesWorked', 'TotalWorkingYears']:
    median_val = df[col].median()
    n_missing = df[col].isnull().sum()
    df[col] = df[col].fillna(median_val)
    print(f'{col}: filled {n_missing} missing values with median = {median_val}')

# Check survey columns for missing
survey_cols = ['EnvironmentSatisfaction', 'JobSatisfaction', 'WorkLifeBalance', 
               'JobInvolvement', 'PerformanceRating']
for col in survey_cols:
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f'{col}: filled {n_missing} missing values with mode = {mode_val}')

print(f'\nRemaining missing values: {df.isnull().sum().sum()}')

NumCompaniesWorked: filled 19 missing values with median = 2.0
TotalWorkingYears: filled 9 missing values with median = 10.0
EnvironmentSatisfaction: filled 25 missing values with mode = 3.0
JobSatisfaction: filled 20 missing values with mode = 4.0
WorkLifeBalance: filled 38 missing values with mode = 3.0

Remaining missing values: 0


---
## 7. Final Merge & Feature Enrichment

In [19]:
# Merge core HR data with attendance features
df_master = df.merge(df_attendance, on='EmployeeID', how='left')

print(f'Master dataset: {df_master.shape[0]:,} rows × {df_master.shape[1]} cols')
print(f'Missing after merge: {df_master.isnull().sum().sum()}')

Master dataset: 4,410 rows × 44 cols
Missing after merge: 0


In [20]:
# Additional derived features

# Tenure group
bins = [0, 1, 3, 5, 10, 50]
labels = ['< 1 year', '1-3 years', '3-5 years', '5-10 years', '10+ years']
df_master['TenureGroup'] = pd.cut(df_master['YearsAtCompany'], bins=bins, labels=labels, right=False)

# Age group
age_bins = [18, 25, 30, 35, 40, 45, 50, 65]
age_labels = ['18-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50+']
df_master['AgeGroup'] = pd.cut(df_master['Age'], bins=age_bins, labels=age_labels, right=False)

# Income quartile within job level
df_master['IncomeQuartile'] = df_master.groupby('JobLevel')['MonthlyIncome'].transform(
    lambda x: pd.qcut(x, 4, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'])
)

# Promotion stagnation flag
df_master['PromotionStagnation'] = (
    (df_master['YearsSinceLastPromotion'] >= 5) & (df_master['YearsAtCompany'] >= 5)
).astype(int)

# Satisfaction composite score (average of all satisfaction metrics)
satisfaction_cols = ['EnvironmentSatisfaction', 'JobSatisfaction', 'WorkLifeBalance']
df_master['SatisfactionScore'] = df_master[satisfaction_cols].mean(axis=1).round(2)

# Career progression ratio
df_master['CareerProgressionRatio'] = (
    df_master['YearsAtCompany'] / df_master['TotalWorkingYears'].clip(lower=1)
).round(2)

# Binary attrition column for modeling
df_master['AttritionFlag'] = (df_master['Attrition'] == 'Yes').astype(int)

print('=== New Features Created ===')
new_features = ['TenureGroup', 'AgeGroup', 'IncomeQuartile', 'PromotionStagnation', 
                'SatisfactionScore', 'CareerProgressionRatio', 'AttritionFlag']
for f in new_features:
    print(f'  ✓ {f}')

=== New Features Created ===
  ✓ TenureGroup
  ✓ AgeGroup
  ✓ IncomeQuartile
  ✓ PromotionStagnation
  ✓ SatisfactionScore
  ✓ CareerProgressionRatio
  ✓ AttritionFlag


In [21]:
# Final dataset overview
print('=' * 60)
print('MASTER DATASET — FINAL OVERVIEW')
print('=' * 60)
print(f'Rows:    {df_master.shape[0]:,}')
print(f'Columns: {df_master.shape[1]}')
print(f'Missing: {df_master.isnull().sum().sum()}')
print(f'\nAttrition rate: {df_master["AttritionFlag"].mean():.2%}')
print(f'\n--- Column List ---')
for i, col in enumerate(df_master.columns, 1):
    dtype = df_master[col].dtype
    nunique = df_master[col].nunique()
    print(f'  {i:2d}. {col:<30s} {str(dtype):<12s} ({nunique:,} unique)')

MASTER DATASET — FINAL OVERVIEW
Rows:    4,410
Columns: 51
Missing: 0

Attrition rate: 16.12%

--- Column List ---
   1. Age                            int64        (43 unique)
   2. Attrition                      object       (2 unique)
   3. BusinessTravel                 object       (3 unique)
   4. Department                     object       (3 unique)
   5. DistanceFromHome               int64        (29 unique)
   6. Education                      int64        (5 unique)
   7. EducationField                 object       (6 unique)
   8. EmployeeID                     int64        (4,410 unique)
   9. Gender                         object       (2 unique)
  10. JobLevel                       int64        (5 unique)
  11. JobRole                        object       (9 unique)
  12. MaritalStatus                  object       (3 unique)
  13. MonthlyIncome                  int64        (1,349 unique)
  14. NumCompaniesWorked             float64      (10 unique)
  15. PercentSalaryH

In [22]:
df_master.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeID,Gender,JobLevel,...,LongDayRate,EngagementTrend,TrendCategory,TenureGroup,AgeGroup,IncomeQuartile,PromotionStagnation,SatisfactionScore,CareerProgressionRatio,AttritionFlag
0,51,No,Travel_Rarely,Sales,6,2,Life Sciences,1,Female,1,...,0.0,0.0022,Stable,1-3 years,50+,Q4 (High),0,3.00,1.00,0
1,31,Yes,Travel_Frequently,Research & Development,10,1,Life Sciences,2,Female,1,...,0.0,0.0121,Stable,5-10 years,30-34,Q2,0,3.00,0.83,1
2,32,No,Travel_Frequently,Research & Development,17,4,Other,3,Male,4,...,0.0,0.0018,Stable,5-10 years,30-34,Q4 (High),0,1.67,1.00,0
3,38,No,Non-Travel,Research & Development,2,5,Life Sciences,4,Male,3,...,0.0,0.0082,Stable,5-10 years,35-39,Q3,1,3.67,0.62,0
4,32,No,Travel_Rarely,Research & Development,10,1,Medical,5,Male,1,...,0.0,0.0052,Stable,5-10 years,30-34,Q1 (Low),0,2.67,0.67,0


---
## 8. Export

In [23]:
# Export master dataset
output_path = '../data/processed/master_dataset.csv'
df_master.to_csv(output_path, index=False)
print(f'✓ Master dataset exported to {output_path}')
print(f'  Size: {df_master.shape[0]:,} rows × {df_master.shape[1]} columns')

# Also export the daily events for notebook 03
events_path = '../data/processed/daily_events.csv'
df_events.to_csv(events_path, index=False)
print(f'✓ Daily events exported to {events_path}')
print(f'  Size: {df_events.shape[0]:,} rows × {df_events.shape[1]} columns')

✓ Master dataset exported to ../data/processed/master_dataset.csv
  Size: 4,410 rows × 51 columns
✓ Daily events exported to ../data/processed/daily_events.csv
  Size: 1,098,090 rows × 14 columns
